<a href="https://colab.research.google.com/github/Manik51/Auto-Video-Editor/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AutoVideoEditor AI — Automated Video Editor
### Professional CapCut-Level Video Editing in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manik51/Auto-Video-Editor/blob/main/main.ipynb)

---
## ⚡ How to Run in Google Colab:
1. **Enable GPU**: Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ **Save**.
2. **Run All**: Press **`Ctrl + F9`** (or click **Runtime** ➔ **Run all**).
3. **Open Studio**: Click the public Gradio link (`https://xxxx.gradio.live`) in Cell 5!
---

In [1]:
# @title 🚀 Step 1: Initialize Environment & Clone Repository
GITHUB_REPO_URL = "https://github.com/Manik51/Auto-Video-Editor.git" #@param {type:"string"}

import os
import sys
import subprocess
import zipfile
import torch

# 1. GPU Check
if torch.cuda.is_available():
    print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"⚡ VRAM: {vram:.2f} GB")
else:
    print("⚠️ No GPU detected. Tip: Go to Runtime -> Change runtime type -> T4 GPU")

# 2. Auto-clone repository into Colab /content if src is missing
IN_COLAB = os.path.exists('/content')

if IN_COLAB and not os.path.exists('src'):
    target_dir = '/content/Auto-Video-Editor'
    if not os.path.exists(target_dir):
        print(f"📥 Cloning repository files from {GITHUB_REPO_URL}...")
        res = subprocess.run(["git", "clone", GITHUB_REPO_URL, target_dir], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"ℹ️ Notice: {res.stderr[:200]}")
            # Check if user uploaded AutoVideoEditor.zip instead
            for zname in ['/content/AutoVideoEditor.zip', 'AutoVideoEditor.zip']:
                if os.path.exists(zname):
                    print(f"📦 Unpacking {zname}...")
                    with zipfile.ZipFile(zname, 'r') as z:
                        z.extractall(target_dir)
                    break

    if os.path.exists(target_dir):
        os.chdir(target_dir)
        if target_dir not in sys.path:
            sys.path.insert(0, target_dir)
        print(f"✅ Active directory set to: {os.getcwd()}")

# 3. Mount Google Drive (Optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/VideoEditor/output', exist_ok=True)
    print("✅ Google Drive mounted at /content/drive")
except Exception:
    pass


✅ GPU Ready: Tesla T4
⚡ VRAM: 14.56 GB
✅ Active directory set to: /content/Auto-Video-Editor
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive


In [2]:
# @title 📦 Step 2: Smart Dependency Check & Installation
import importlib.util
import subprocess
import sys

print("📦 Fixing OpenCV installation first...")
# Force reinstall standard opencv to fix missing attributes like CascadeClassifier
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-python-headless', 'opencv-contrib-python'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'opencv-python'], check=False)

print("📦 Checking core editing dependencies...")

CORE_PACKAGES = {
    'moviepy': 'moviepy==1.0.3',
    'cv2': 'opencv-python',
    'ffmpeg': 'ffmpeg-python',
    'scenedetect': 'scenedetect[opencv]',
    'whisper': 'openai-whisper',
    'soundfile': 'soundfile',
    'noisereduce': 'noisereduce',
    'gradio': 'gradio',
}

for mod, pkg in CORE_PACKAGES.items():
    if importlib.util.find_spec(mod) is None:
        print(f"📦 Installing {pkg}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg], check=False)
        print(f"✅ {mod:<14} : Installed")
    else:
        print(f"✅ {mod:<14} : Ready")

# Optional modules (fault-tolerant so notebook never crashes)
OPTIONAL_PACKAGES = {
    'librosa': 'librosa',
    'rembg': 'rembg',
    'gfpgan': 'gfpgan',
}

print("\n📦 Checking optional AI enhancers...")
for mod, pkg in OPTIONAL_PACKAGES.items():
    if importlib.util.find_spec(mod) is None:
        try:
            print(f"📦 Attempting optional {pkg}...")
            res = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg], capture_output=True, text=True)
            if res.returncode == 0:
                print(f"✅ {mod:<14} : Installed")
            else:
                print(f"ℹ️ Optional {mod} skipped (Core editor runs 100% without it).")
        except Exception:
            print(f"ℹ️ Optional {mod} skipped.")
    else:
        print(f"✅ {mod:<14} : Ready")

print("\n🚀 All dependencies verified successfully!")

📦 Fixing OpenCV installation first...
📦 Checking core editing dependencies...
✅ moviepy        : Ready
✅ cv2            : Ready
✅ ffmpeg         : Ready
✅ scenedetect    : Ready
✅ whisper        : Ready
✅ soundfile      : Ready
✅ noisereduce    : Ready
✅ gradio         : Ready

📦 Checking optional AI enhancers...
✅ librosa        : Ready
✅ rembg          : Ready
📦 Attempting optional gfpgan...
ℹ️ Optional gfpgan skipped (Core editor runs 100% without it).

🚀 All dependencies verified successfully!


In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
if os.path.exists('/content/Auto-Video-Editor'):
    sys.path.insert(0, '/content/Auto-Video-Editor')

# Fix for numpy/scipy incompatibility
!pip install --upgrade --force-reinstall numpy scipy

from src.installer import check_system_resources
from src.scene_detector import SceneDetector
from src.audio_analyzer import AudioAnalyzer
from src.silence_remover import SilenceRemover
from src.color_grader import ColorGrader
from src.caption_generator import CaptionGenerator
from src.transition_engine import TransitionEngine
from src.effects_engine import EffectsEngine
from src.face_enhancer import FaceEnhancer
from src.audio_enhancer import AudioEnhancer
from src.renderer import VideoRenderer
from src.pipeline import VideoPipeline

print("✅ All AutoVideoEditor modules loaded successfully!")
check_system_resources()


  Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.3
    Uninstalling numpy-2.5.3:
      Successfully uninstalled numpy-2.5.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.18.1
    Uninstalling scipy-1.18.1:
      Successfully uninstalled scipy-1.18.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
albumentations 2.0.8 requires opencv-pytho

✅ All AutoVideoEditor modules loaded successfully!
🚀 CUDA GPU Ready: Tesla T4 (14.56 GB VRAM)
🎬 FFmpeg Ready: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


{'gpu_available': True,
 'gpu_name': 'Tesla T4',
 'gpu_memory_gb': 14.56,
 'ffmpeg_available': True,
 'ffmpeg_version': 'ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers'}

In [4]:
# @title 📥 Step 4: Download AI Models & Fonts
from src.installer import download_models, download_fonts

print("📥 Downloading AI models (Whisper, GFPGAN) & typography assets...")
download_fonts()
download_models()
print("✅ All AI models and fonts are cached and ready!")


📥 Downloading AI models (Whisper, GFPGAN) & typography assets...
  ✅ Font Montserrat-Bold.ttf is already available.
  ✅ Font Roboto-Bold.ttf is already available.
  ✅ GFPGAN v1.4 weights already cached.
📥 Pre-loading Whisper base model...
  ✅ Whisper base model loaded successfully.
✅ All AI models and fonts are cached and ready!


In [5]:
# @title 🎬 Step 5: Launch AutoVideoEditor Web Studio
import subprocess
import sys
from ui.gradio_app import launch_ui

print("🧹 Cleaning up any conflicting background processes on port 7860...")
try:
    # Find and terminate process running on port 7860 to release it
    subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True)
    print("✅ Port 7860 is now free and ready!")
except Exception as e:
    print(f"ℹ️ Port cleanup skipped: {e}")

print("\n=======================================================")
print("🎉 LAUNCHING VIDEO EDITOR WEB STUDIO...")
print("👉 Click the 'Running on public URL' link below!")
print("=======================================================\n")

try:
    launch_ui(share=True, server_port=7860)
except Exception as e:
    print(f"\n❌ Gradio Launch Failed! Error: {e}", file=sys.stderr)
    import traceback
    traceback.print_exc()

🧹 Cleaning up any conflicting background processes on port 7860...
✅ Port 7860 is now free and ready!

🎉 LAUNCHING VIDEO EDITOR WEB STUDIO...
👉 Click the 'Running on public URL' link below!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c2cba0d78f1202140.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# @title 💻 Step 6: (Alternative) CLI Batch Processing
import os
from src.pipeline import VideoPipeline

sample_video_path = 'sample_raw_video.mp4'
if os.path.exists(sample_video_path):
    print(f"🎬 Processing {sample_video_path} via CLI...")
    pipeline = VideoPipeline(
        input_path=sample_video_path,
        preset="YouTube Vlog",
        options={
            "auto_captions": True,
            "color_grade": True,
            "remove_silence": True,
            "face_enhance": False,
            "beat_sync": False,
        },
    )
    results = pipeline.run()
    print(f"🎉 Final edited video saved at: {results['output_path']}")
else:
    print(f"ℹ️ Place a video at '{sample_video_path}' to test CLI processing.")


ℹ️ Place a video at 'sample_raw_video.mp4' to test CLI processing.


In [7]:
import subprocess
import time
import sys

print("🔍 Checking background running processes (especially FFmpeg and Python)...\n")

def check_live_logs():
    # Check active ffmpeg or python tasks
    try:
        ps_output = subprocess.check_output(["ps", "aux"], text=True)
        active_processes = [line for line in ps_output.split('\n') if 'ffmpeg' in line or 'gradio_app' in line or 'pipeline' in line]
        if active_processes:
            print("🔥 Active Background Processes:")
            for p in active_processes:
                print(f"  {p[:120]}")
        else:
            print("ℹ️ No active background FFmpeg rendering detected right now.")
    except Exception as e:
        print(f"Error checking processes: {e}")

    # Checking the last 30 lines of the system logs or gradio log if any exist
    print("\n--- Live System State ---")
    !tail -n 30 /root/.gradio/gradio.log 2>/dev/null || echo "ℹ️ No Gradio log file found yet. Run Step 5 first!"

check_live_logs()

🔍 Checking background running processes (especially FFmpeg and Python)...

ℹ️ No active background FFmpeg rendering detected right now.

--- Live System State ---
ℹ️ No Gradio log file found yet. Run Step 5 first!
